In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import calendar

START_Y = 2024
END_Y = 2025
roi_path = 'roi_name_in_csv_filename'  # Must be stored in local directory, in folder 'results'

df = pd.read_csv(f'./results/reduced_sc_{roi_path}_{START_Y}_{END_Y}.csv')

# Preprocess dataset

In [ ]:
# Rename fields
df = df.rename(columns={
    'flag_all_mean_mean': 'gap_all',
    'flag_dt_mean_mean': 'gap_ml',
    'flag_all_mean_sum': 'gap_all_sum',
    'flag_dt_mean_sum': 'gap_ml_sum',
    'sc_mean_sum': 'sca',
    'sc_mean_mean': 'fsc',
    'month_mean': 'month',
    'year_mean': 'year',
    't2mHres_mean_mean': 't2mHres_mean',
    'precipHres_sum_mean': 'precipHres_sum',
    'tp_sum_sum': 'tp',
    'fp_sum_sum': 'fp',
    'tn_sum_sum': 'tn',
    'fn_sum_sum': 'fn'
})

# Convert Snow Cover Area to km²
df['sca'] = df['sca'] * 0.01

# Calculate the total number of grid cells in the ROI per month
df['count'] = df['year_sum']/df['year']

# Calculate skill metrics
tp, tn, fp, fn = df['tp'], df['tn'], df['fp'], df['fn']
df['n'] = tp + tn + fp + fn
df['accuracy'] = (tp + tn) / df['n']
df['overestimation'] = fp / df['n']
df['underestimation'] = fn / df['n']

# Data fractions in reconstruction
df['clear_sky'] = (df['count'] - df['gap_all_sum'])*100/df['count'] # clear sky
df['gap_dt'] = (df['gap_all_sum'] - df['gap_ml_sum'])*100/df['count']
df['gap_ml'] = df['gap_ml_sum']*100/df['count']

# Evaluation summary

In [ ]:
eval_sum_stats = pd.Series({
    'count_sum': df['count'].sum(),
    'clear_sky_sum_num': df['count'].sum() - df['gap_all_sum'].sum(),
    'clear_sky_sum': (df['count'].sum() - df['gap_all_sum'].sum())*100/df['count'].sum(),
    'gap_dt_sum': (df['gap_all_sum'].sum() - df['gap_ml_sum'].sum())*100/df['count'].sum(),
    'gap_ml_sum': df['gap_ml_sum'].sum()*100/df['count'].sum(),   
    'accuracy_sum': (df['tp'].sum() + df['tn'].sum()) / df['n'].sum(),
    'overestimation_sum': df['fp'].sum() / df['n'].sum(),
    'underestimation_sum': df['fn'].sum() / df['n'].sum()
})

round(eval_sum_stats, 2)

# Plot monthly proportions of pixels

In [ ]:
df['month'] = df['month'].astype(int)
df['Month'] = df['month'].apply(lambda x: calendar.month_abbr[x])
unique_month_nums = sorted(df['month'].unique())
ordered_months = [calendar.month_abbr[m] for m in unique_month_nums]
stack_cols = ['clear_sky', 'gap_dt', 'gap_ml']
df_monthly = df.groupby('Month')[stack_cols].sum()
df_monthly = df_monthly.reindex(ordered_months)

plt.figure(figsize=(10, 6))
df_monthly.plot(
    kind='bar',
    stacked=True,
    figsize=(7, 4),
)

plt.title(f"{roi_path} {START_Y}-{END_Y}\nClear sky={round(df['clear_sky'].mean(), 1)}%\nDecision tree gapfilled={round(df['gap_dt'].mean(), 1)}%\nMachine learning gapfilled={round(df['gap_ml'].mean(), 1)}%")
plt.ylabel("Percentage of grid cells")
plt.xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Plot snowMapper model skill metrics

In [ ]:
metrics_order = ['accuracy', 'overestimation', 'underestimation']
df['Month'] = df['month'].astype(int).apply(lambda x: calendar.month_abbr[x])
unique_month_nums = sorted(df['month'].astype(int).unique())
month_filter = [calendar.month_abbr[m] for m in unique_month_nums]
df_filtered = df[df['Month'].isin(month_filter)].copy()
df_filtered['MonthLabel'] = df_filtered.apply(
    lambda row: f"{row['Month']}\n(n={int(row['n'])})", axis=1
)

df_melted = df_filtered.melt(
    id_vars='MonthLabel',
    value_vars=[m for m in metrics_order if m in df_filtered.columns],
    var_name='Metric',
    value_name='Value'
)

heatmap_data = df_melted.pivot(index='Metric', columns='MonthLabel', values='Value')
month_label_map = df_filtered.set_index('Month')['MonthLabel'].to_dict()

metric_means = heatmap_data.mean(axis=1)
heatmap_data.index = [
    f"{metric}\n(mean: {metric_means[metric]:.2f})"
    for metric in heatmap_data.index
]

plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data,
    cmap='YlGnBu',
    cbar=True,
    vmin=0,
    vmax=1,
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    cbar_kws={'label': 'Metric Value'}
)

total_n = int(df_filtered['n'].sum())
plt.title(f'{roi_path} {START_Y}-{END_Y}\nValidation against n={total_n} clear-sky observations')
plt.xlabel('')
plt.ylabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()